### Imports and Hyperparameter Settings

In [ ]:
%load_ext autoreload
%autoreload 2

from multimodal_mazes.evolution.module_evolution.module_genome import Module
    
import numpy as np
import unittest
import copy

In [ ]:
HYPERPARAMETERS = {
    'n_inputs': 8,
    'n_outputs': 4,
    'n_modules': 4,
    'weight_sharing': False,
    'uniform_weights': False, # For weight sharing
    'connectivity': 'UNCONNECTED', # Options: 'FULLY CONNECTED', 'UNCONNECTED', 'SPARSE', 'RANDOM'
    'connection_density' : {'input_density': None, 'output_density': None}, # For 'RANDOM' connectivity
    'population_size': 80,
    'top_genomes': 20, 
    'mutation_rate': 0.8, 
    'crossover_rate': 0.5,
}

## Genome

### Unit Testing

In [ ]:
class TestGenome(unittest.TestCase):
    def setUp(self):
        """Set up the test case."""
        self.genome = Module(module_id=0, hyperparameters=HYPERPARAMETERS)

    def test_initialization(self):
        """
        Test the initialization of the genome.
        Tests:
            Module ID is set correctly.
            Node rules are initialized.
            Input, hidden, and output nodes are created.
            Connection rules are initialized.
            Compilation flag is set.
        """
        # Check module id
        self.assertEqual(self.genome.module_id, 0)

        # Check node rules
        self.assertEqual(len(self.genome.node_rules), 3)
        self.assertEqual(len(self.genome.nodes_in), 2)
        self.assertEqual(len(self.genome.nodes_hid), 0)
        self.assertEqual(len(self.genome.nodes_out), 1)
        self.assertEqual(len(self.genome.grouped_in), 2)
        self.assertEqual(len(self.genome.grouped_hid), 0)
        self.assertEqual(len(self.genome.grouped_out), 1)

        # Check connection rules 
        self.assertEqual(self.genome.conn_in_rules, [])
        self.assertEqual(self.genome.conn_hid_rules, [])
        self.assertEqual(self.genome.conn_out_rules, [])

        # Check compile flag
        self.assertEqual(self.genome.compile_flag, [1, 1, 1])
        
    def test_forward_pass(self):
        """
        Test the forward pass of the genome.
        Tests:
            - Output is a numpy array.
            - Output shape is correct.
        """
        input_vec = np.ones(self.genome.n_inputs)
        output_vec = self.genome.forward_pass(input_vec)

        # Check output
        self.assertIsInstance(output_vec, np.ndarray)
        self.assertEqual(output_vec.shape[0], self.genome.n_outputs)

    def test_crossover(self):
        """
        Test the crossover of the genome.
        Tests:
            Child genome is created.
            Child genome has correct ID.
            Child genome has compile flag set.
            Child genome inherits properties from parents.
        """
        parent_1 = Module(module_id=1,hyperparameters=HYPERPARAMETERS)
        parent_2 = Module(module_id=2, hyperparameters=HYPERPARAMETERS)
        
        for _ in range(10):
            parent_1.mutate()
            parent_2.mutate()

        child = parent_1.crossover(new_id=3, parent_2=parent_2)

        # Check child genome properties
        self.assertIsInstance(child, Module)
        self.assertEqual(child.module_id, 3)
        self.assertEqual(child.compile_flag, [1, 1, 1])

        # Check child correctly inherits from parents
        if child.hidden_ids:
            self.assertNotEqual(child.node_rules, parent_1.node_rules)
            self.assertNotEqual(child.conn_hid_rules, parent_1.conn_hid_rules)
            self.assertNotEqual(child.conn_hid_rules, parent_2.conn_hid_rules)
        if parent_1.n_outputs == 1 and parent_2.n_outputs == 1:
            self.assertEqual(child.conn_out_rules, parent_1.conn_out_rules)
            self.assertNotEqual(child.conn_out_rules, parent_2.conn_out_rules)
        else:
            self.assertNotEqual(child.conn_out_rules, parent_1.conn_out_rules)
            self.assertNotEqual(child.conn_out_rules, parent_2.conn_out_rules)
        
    def test_mutation(self):
        """
        Test the mutation of the genome.
        Tests:
            Mutation changes the genome's structure.
        """
        mut_genome = Module(module_id=5, hyperparameters=HYPERPARAMETERS)
        org_node_rules = copy.deepcopy(mut_genome.node_rules)
        org_conn_in_rules = copy.deepcopy(mut_genome.conn_in_rules)
        org_conn_hid_rules = copy.deepcopy(mut_genome.conn_hid_rules)
        org_conn_out_rules = copy.deepcopy(mut_genome.conn_out_rules)

        for _ in range(10):
            mut_genome.mutate()

        # Check that mutation has occurred
        if mut_genome.hidden_ids:
            self.assertNotEqual(mut_genome.node_rules, org_node_rules)
            self.assertNotEqual(mut_genome.conn_hid_rules, org_conn_hid_rules)
        self.assertNotEqual(mut_genome.conn_in_rules, org_conn_in_rules)
        self.assertNotEqual(mut_genome.conn_out_rules, org_conn_out_rules)

    def test_clone(self):
        """
        Test the cloning of the genome.
        Tests:
            Cloned genome is created.
            Cloned genome has correct ID.
            Cloned genome has compile flag set.
        """
        cloned_genome = self.genome.clone(new_id=4)

        # Check cloned genome properties
        self.assertIsInstance(cloned_genome, Module)
        self.assertEqual(cloned_genome.module_id, 4)
        self.assertEqual(cloned_genome.compile_flag, [1, 1, 1])
        
        # Check cloned genome matches original
        self.assertEqual(self.genome.n_hidden_curr, cloned_genome.n_hidden_curr)
        self.assertEqual(self.genome.n_hidden_total, cloned_genome.n_hidden_total)
        self.assertEqual(self.genome.hidden_ids, cloned_genome.hidden_ids)
        for rule1, rule2 in zip(cloned_genome.node_rules, self.genome.node_rules):
            self.assertTupleEqual(rule1, rule2)
        for rule1, rule2 in zip(cloned_genome.conn_in_rules, self.genome.conn_in_rules):
            self.assertTupleEqual(rule1, rule2)
        for rule1, rule2 in zip(cloned_genome.conn_hid_rules, self.genome.conn_hid_rules):
            self.assertTupleEqual(rule1, rule2)
        for rule1, rule2 in zip(cloned_genome.conn_out_rules, self.genome.conn_out_rules):
            self.assertTupleEqual(rule1, rule2)

In [ ]:
module_suite = unittest.TestLoader().loadTestsFromTestCase(TestGenome)
unittest.TextTestRunner(verbosity=2).run(module_suite)

### Output Inspection

#### Initialisation

In [ ]:
genome = Module(module_id=0, hyperparameters=HYPERPARAMETERS)

print("Genome ID:", genome.module_id)

print("Node Rules:")
for rule in genome.node_rules:
    print(rule)

print("\nInput Connect Rules:")
for rule in genome.conn_in_rules:
    print(rule)

print("\nOutput Connect Rules:")
for rule in genome.conn_out_rules:
    print(rule)

genome.plot_genome()

#### Mutation

In [ ]:
genome_mut = Module(module_id=0, hyperparameters=HYPERPARAMETERS)

print("Original Input Connection Rules:")
for rule in genome_mut.conn_in_rules:
    print(rule)

print("Original Output Connection Rules:")
for rule in genome_mut.conn_out_rules:
    print(rule)

# genome_mut.plot_genome()
for _ in range(10):
    genome_mut.mutate()
# genome_mut.plot_genome()

print("\nMutated Input Connection Rules:")
for rule in genome_mut.conn_in_rules:
    print(rule)
print("\nMutated Output Connection Rules:")
for rule in genome_mut.conn_out_rules:
    print(rule)

#### Forward Pass

In [ ]:
genome_fp = Module(module_id=0, hyperparameters=HYPERPARAMETERS)
genome_fp.plot_genome()

In [ ]:
input_vec = np.array([0.0, 1.0])
print("Input Vector:", input_vec)

output_vec = genome_fp.forward_pass(input_vec)
print("Output Vector:", output_vec)

input_vec = np.array([0.0, 0.0])
print("Input Vector:", input_vec)

output_vec = genome_fp.forward_pass(input_vec)
print("Output Vector:", output_vec)

#### Compile

In [ ]:
genome_c = Module(module_id=0, hyperparameters=HYPERPARAMETERS)
genome_c.plot_genome()

In [ ]:
genome_c.compile_flag = [1, 1, 1]
genome_c.compile_rules()

print("Input connectivity rules:")
print(genome_c.grouped_in)

print("Hidden connectivity rules:")
print(genome_c.grouped_hid)

print("Output connectivity rules:")
print(genome_c.grouped_out)